In [ ]:
# PART 1 — PDF INGESTION & PREPROCESSING
# -------------------------------
# Task 1: Load PDF Documents
# -------------------------------

# Step 1: Import libraries
from langchain_community.document_loaders import PyPDFLoader

# Step 2: Load PDF
pdf_path = "data/rag_paper.pdf"
loader = PyPDFLoader(pdf_path)

documents = loader.load()

# Step 3: Print metadata
print("Total Pages:", len(documents))
print("\nSample Content:\n")
print(documents[0].page_content[:500])

Total Pages: 100

Sample Content:

GPT-4 Technical Report
OpenAI∗
Abstract
We report the development of GPT-4, a large-scale, multimodal model which can
accept image and text inputs and produce text outputs. While less capable than
humans in many real-world scenarios, GPT-4 exhibits human-level performance
on various professional and academic benchmarks, including passing a simulated
bar exam with a score around the top 10% of test takers. GPT-4 is a Transformer-
based model pre-trained to predict the next token in a document. Th


In [4]:
# -------------------------------
# Task 2: Text Splitting
# -------------------------------

# Step 1: Import splitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Step 2: Configure splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150
)

# Step 3: Split documents
chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 535


## Part 1: PDF Ingestion and Preprocessing

In this section, the PDF document is loaded and processed into smaller text chunks. Chunking helps improve retrieval accuracy by allowing the system to search relevant portions instead of the entire document.

In [5]:

# PART 2 — EMBEDDINGS & VECTOR STORE


# -------------------------------
# Task 3: Create Embeddings
# -------------------------------

# Step 1: Import embedding model
from langchain_community.embeddings import HuggingFaceEmbeddings

# Step 2: Load FREE embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# -------------------------------
# Task 4: Vector Store Setup
# -------------------------------

# Step 1: Import FAISS
from langchain_community.vectorstores import FAISS

# Step 2: Create vector database
vectorstore = FAISS.from_documents(chunks, embedding_model)

# Step 3: Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

d:\A31_Conversational-PDF-QA\venv311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## Part 2: Embeddings and Vector Database

Here, text chunks are converted into vector embeddings using a local embedding model. These embeddings are stored in a FAISS vector database to enable semantic search.

In [ ]:

# PART 3 — PROMPT TEMPLATE


from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a PDF Question Answering assistant.

Use ONLY the provided CONTEXT to answer.
Do NOT use outside knowledge.

If answer is not in context, say:
"I don't know based on the document."

CONTEXT:
{context}
"""),

    MessagesPlaceholder(variable_name="chat_history"),

    ("human", "{question}")
])

In [8]:

# PART 4 — CONVERSATIONAL RAG CHAIN


# Step 1: Load FREE LLM (Ollama)
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="gemma:2b")  

In [9]:
# Step 2: Chat history storage
chat_history = []

def conversational_rag(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([d.page_content for d in docs])

    messages = prompt.format_messages(
        context=context,
        chat_history=chat_history,
        question=question
    )

    response = llm.invoke(messages)

    chat_history.append(("human", question))
    chat_history.append(("ai", response.content))

    return response.content

In [10]:
# -------------------------------
# Task 8: Trimming Chat History
# -------------------------------

MAX_MESSAGES = 6

def trim_history():
    global chat_history
    if len(chat_history) > MAX_MESSAGES:
        chat_history = chat_history[-MAX_MESSAGES:]

In [12]:
def conversational_rag(question):

    docs = retriever.invoke(question)

    print("\n--- Retrieved Context Preview ---\n")
    print(docs[0].page_content[:400])   # DEBUG

    context = "\n\n".join([d.page_content for d in docs])

    messages = prompt.format_messages(
        context=context,
        chat_history=chat_history,
        question=question
    )

    response = llm.invoke(messages)

    chat_history.append(("human", question))
    chat_history.append(("ai", response.content))

    return response.content


# PART 5 — FOLLOW-UP Q&A TESTING


print(conversational_rag("What is the main contribution described in the abstract?"))
print(conversational_rag("Explain that in simple words"))

trim_history()


--- Retrieved Context Preview ---

∗Please cite this work as “OpenAI (2023)". Full authorship contribution statements appear at the end of the
document. Correspondence regarding this technical report can be sent to gpt4-report@openai.comarXiv:2303.08774v6  [cs.CL]  4 Mar 2024
The main contribution of the paper is to explore the effectiveness of different neural network architectures for tasks related to the human body and the natural world.

--- Retrieved Context Preview ---

content.
- (R) None of the above.
Your response should start with only the single character "A" or "B" or "C" or "D" or "E" or "F" or "G" or "H" or "I" or "J" or
"K" or "L" or "M" or "N" or "O" or "P" or "Q" or "R" (without quotes or punctuation) on its own line followed by an explanation
of your answer on the next line. Your explanation should take the reader through your reasoning step-by-step, 
The paper provides insights into the strengths and weaknesses of different neural network architectures for the task 

In [ ]:

# PART 6 — FINAL CHATBOT APPLICATION


print("Conversational PDF Chatbot Ready (type exit to stop)\n")

while True:
    query = input("You: ")

    if query.lower() == "exit":
        break

    answer = conversational_rag(query)
    trim_history()

    print("AI:", answer)

Conversational PDF Chatbot Ready (type exit to stop)


--- Retrieved Context Preview ---

H System Card
The System Card [84, 85] for GPT-4 is appended to this document.
40
AI: GPT-4 is a large language model that has been trained on a massive dataset of text and code.

--- Retrieved Context Preview ---

model mitigations on datasets in multiple languages.
•Ensure that safety assessments cover emergent risks: As models get more capable, we
should be prepared for emergent capabilities and complex interactions to pose novel safety issues.
It’s important to develop evaluation methods that can be targeted at advanced capabilities that
could be particularly dangerous if they emerged in future models, w
AI: The context does not specify any limitations of GPT-4, so I cannot answer this question from the context.


Observations & Insights
1. Difference Between PDF QA and Conversational QA

While testing the system, I noticed that a normal PDF QA setup treats every question independently and does not remember previous interactions. In contrast, the conversational QA system was able to retain context from earlier questions. Because of this, follow-up queries like “explain that simply” or “what about its limitations?” worked naturally without repeating the full question. This made the interaction feel closer to a real conversation rather than a search tool.

2. Importance of Message History

Maintaining chat history played a major role in improving answer quality. When message history was included, the model could understand references to earlier responses and generate more coherent explanations. Without history, follow-up questions often produced incomplete or unrelated answers. This shows that conversational memory is essential for multi-turn reasoning in RAG systems.

3. Memory vs Performance Trade-off

During experimentation, I observed that increasing chat history improved contextual understanding but also slowed response time slightly. More history means more tokens are processed by the model. Therefore, keeping unlimited history is not efficient. A balanced approach is required to maintain both performance and response quality.

4. Impact of History Trimming

Implementing history trimming helped keep the chatbot stable during longer conversations. It prevented excessive context growth and reduced latency. Although some older conversation details were lost, the chatbot still maintained enough recent context to answer follow-up questions effectively. This demonstrated the practical need for controlled memory management in conversational AI systems.